In [54]:
import os
import re
from collections import Counter, defaultdict

In [55]:
DIR_DATA = "data"
corpus_path = os.path.join(DIR_DATA, "cleaned_corpus.txt")

with open(corpus_path, "r", encoding="utf-8") as f:
    text = f.read().lower()

# Keep words and sentence-end tokens
tokens = re.findall(r"\b\w+\b|[.!?]", text)

print("Number of tokens:", len(tokens))
print(tokens[:30])

Number of tokens: 12207411
['the', 'modern', 'prometheus', 'by', 'mary', 'shelley', 'contents', 'letter', 'letter', 'letter', 'letter', 'letter', 'mrs', '.', 'st', '.', 'dec', '.', '.', 'you', 'will', 'rejoice', 'to', 'hear', 'that', 'no', 'disaster', 'has', 'accompanied', 'the']


In [56]:
unigram_counts = Counter(tokens)
bigram_counts = Counter(zip(tokens[:-1], tokens[1:]))
trigram_counts = Counter(zip(tokens[:-2], tokens[1:-1], tokens[2:]))

V = len(unigram_counts)

print("Vocabulary size:", V)
print("Number of unique bigrams:", len(bigram_counts))
print("Number of unique trigrams:", len(trigram_counts))

Vocabulary size: 25280
Number of unique bigrams: 1615643
Number of unique trigrams: 6083536


In [57]:
def bigram_prob(w1, w2):
    numerator = bigram_counts.get((w1, w2), 0)
    denominator = unigram_counts.get(w1, 0)

    if denominator == 0:
        return 0.0

    return numerator / denominator

In [58]:
print(bigram_prob("of", "the"))

0.21782327754381814


In [59]:
def bigram_prob_laplace(w1, w2):
    numerator = bigram_counts.get((w1, w2), 0) + 1
    denominator = unigram_counts.get(w1, 0) + V
    return numerator / denominator

In [60]:
print(bigram_prob_laplace("of", "the"))
print(bigram_prob_laplace("of", "jesus"))

0.20479378625336353
0.00014199603831053113


In [61]:
def trigram_prob(w1, w2, w3):
    numerator = trigram_counts.get((w1, w2, w3), 0)
    denominator = bigram_counts.get((w1, w2), 0)

    if denominator == 0:
        return 0.0

    return numerator / denominator

In [62]:
print(trigram_prob("in", "the", "house"))

0.0077874116891457935


In [63]:
def trigram_prob_laplace(w1, w2, w3):
    numerator = trigram_counts.get((w1, w2, w3), 0) + 1
    denominator = bigram_counts.get((w1, w2), 0) + V
    return numerator / denominator

In [64]:
print(trigram_prob_laplace("in", "the", "house"))
print(trigram_prob_laplace("in", "the", "book"))

0.0055504796710826865
0.0005025125628140704


In [65]:
bigram_next = defaultdict(Counter)
for (w1, w2), count in bigram_counts.items():
    bigram_next[w1][w2] = count

trigram_next = defaultdict(Counter)
for (w1, w2, w3), count in trigram_counts.items():
    trigram_next[(w1, w2)][w3] = count

In [66]:
def predict_next_bigram(sequence):
    if isinstance(sequence, str):
        sequence = sequence.lower().split()

    if len(sequence) == 0:
        return None

    last_word = sequence[-1]

    if last_word not in bigram_next:
        return None

    return bigram_next[last_word].most_common(1)[0][0]

In [67]:
print(predict_next_bigram("of"))
print(predict_next_bigram("jesus"))

the
christ


In [68]:
def predict_next_trigram(sequence):
    if isinstance(sequence, str):
        sequence = sequence.lower().split()

    if len(sequence) < 2:
        return None

    context = (sequence[-2], sequence[-1])

    if context not in trigram_next:
        return None

    return trigram_next[context].most_common(1)[0][0]

In [69]:
print(predict_next_trigram("the jesus"))
print(predict_next_trigram("in jesus"))

of
christ


In [70]:
def generate_bigram(start_text, max_steps=20):
    words = start_text.lower().split()

    for _ in range(max_steps):
        next_word = predict_next_bigram(words)

        if next_word is None:
            break

        words.append(next_word)

        if next_word in [".", "!", "?"]:
            break

    return " ".join(words)

In [71]:
def generate_trigram(start_text, max_steps=20):
    words = start_text.lower().split()

    for _ in range(max_steps):
        next_word = predict_next_trigram(words)

        if next_word is None:
            break

        words.append(next_word)

        if next_word in [".", "!", "?"]:
            break

    return " ".join(words)

In [72]:
print("Bigram generation:")
print(generate_bigram("to", max_steps=15))

print("\nTrigram generation:")
print(generate_trigram("find the", max_steps=15))

Bigram generation:
to the and the and the and the and the and the and the and the

Trigram generation:
find the way of the and the and the and the and the and the and the


In [73]:
print(generate_trigram("in jesus", max_steps=15))
print(generate_trigram("in god", max_steps=15))

in jesus christ .
in god and the and the and the and the and the and the and the and


In [74]:
w1, w2 = "of", "the"
print("Bigram unsmoothed:", bigram_prob(w1, w2))
print("Bigram Laplace   :", bigram_prob_laplace(w1, w2))

w1, w2, w3 = "in", "the", "house"
print("Trigram unsmoothed:", trigram_prob(w1, w2, w3))
print("Trigram Laplace   :", trigram_prob_laplace(w1, w2, w3))

Bigram unsmoothed: 0.21782327754381814
Bigram Laplace   : 0.20479378625336353
Trigram unsmoothed: 0.0077874116891457935
Trigram Laplace   : 0.0055504796710826865


 With the bigram model, the text quickly becomes repetitive. For example, starting with “to” produced the sequence “to the and the and the…”. This happens because the model always chooses the most frequent next word, and combinations like “the and” and “and the” appear very often in the corpus. The trigram model sometimes gives a slightly better start because it uses two previous words as context. For instance, starting with “find the” produced “find the way of the…”, which sounds like a normal phrase at first, but after a few words the sentence again becomes repetitive. Another example is “in jesus christ.”, which shows that the model can learn short fixed phrases from the corpus. However, starting with “in god” produced “in god and the and the…”, which again becomes repetitive and less meaningful. The probability values also show the expected effect of Laplace smoothing. The unsmoothed probability of “the” after “of” is quite high because “of the” is very common in English texts, while the smoothed probability is slightly lower because the model distributes some probability across all possible words. Overall, the trigram model produced slightly more natural fragments than the bigram model, but both models often generate repetitive text because they only use local word frequencies and do not understand grammar or meaning.
